<a href="https://colab.research.google.com/github/Subah-Zarin/Wall-Decoration-Pattern-Analysis-/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CAMPUS WALL DECORATION
# BEST-PRACTICE PREPROCESSING PIPELINE
#
# DATA SOURCE:
#   Google Sheet -> metadata / labels
#   Google Drive -> images
#
# MAIN FEATURES:
#   ✅ Image-ID matching
#   ✅ Missing image detection
#   ✅ Corrupted image detection
#   ✅ Duplicate image-ID detection
#   ✅ Duplicate/near-duplicate image check
#   ✅ Description-based event/decor grouping
#   ✅ Group-aware train/val/test split
#   ✅ 80/10/10 split
#   ✅ 224x224 resize
#   ✅ RGB conversion
#   ✅ ImageNet normalization
#   ✅ Training-only augmentation
#   ✅ No original image modification
#   ✅ Reusable split CSV files
#   ✅ Before/after visualization
#   ✅ Augmentation visualization
#   ✅ Final statistics
# ============================================================


# ============================================================
# CELL 1 — INSTALL PACKAGES
# ============================================================

!pip -q install \
    pandas \
    numpy \
    pillow \
    matplotlib \
    scikit-learn \
    torch \
    torchvision \
    sentence-transformers \
    gspread \
    google-api-python-client


# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import re
import json
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False

import torch

from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")


# ============================================================
# CELL 3 — USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# Google Sheet
# ------------------------------------------------------------

GOOGLE_SHEET_NAME = "Softcomp Annotation"

SHEET_NAME = "Sheet1"


# ------------------------------------------------------------
# Google Drive image folder
# ------------------------------------------------------------

IMAGE_FOLDER = (
    "/content/drive/MyDrive/"
    "Wall decoration images"
)


# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUTPUT_FOLDER = (
    "/content/drive/MyDrive/"
    "Campus_Wall_Preprocessing"
)


# ------------------------------------------------------------
# Dataset split
# ------------------------------------------------------------

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42


# ------------------------------------------------------------
# Description grouping threshold
#
# Higher = stricter grouping
# Lower = more images grouped together
#
# Start with 0.80
# ------------------------------------------------------------

DESCRIPTION_SIMILARITY_THRESHOLD = 0.80


# ------------------------------------------------------------
# Supported images
# ------------------------------------------------------------

SUPPORTED_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)


# ============================================================
# CELL 4 — REPRODUCIBILITY
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("✅ Seed:", SEED)


# ============================================================
# CELL 5 — CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

print(
    "✅ Output folder:",
    OUTPUT_FOLDER
)


# ============================================================
# CELL 6 — GOOGLE AUTHENTICATION
# ============================================================

from google.colab import auth

auth.authenticate_user()

import google.auth
import gspread

SCOPES = [
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/spreadsheets"
]

creds, _ = google.auth.default(
    scopes=SCOPES
)

gc = gspread.authorize(creds)

print("✅ Google account connected")


# ============================================================
# CELL 7 — GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print("✅ Google Drive mounted")


# ============================================================
# CELL 8 — CHECK IMAGE FOLDER
# ============================================================

if not os.path.exists(
    IMAGE_FOLDER
):

    raise FileNotFoundError(
        f"\n❌ Image folder not found:\n"
        f"{IMAGE_FOLDER}"
    )

print(
    "✅ Image folder found"
)


# ============================================================
# CELL 9 — CONNECT GOOGLE SHEET
# ============================================================

spreadsheet = gc.open(
    GOOGLE_SHEET_NAME
)

worksheet = spreadsheet.worksheet(
    SHEET_NAME
)

print(
    "✅ Google Sheet connected"
)

print(
    "Spreadsheet:",
    spreadsheet.title
)

print(
    "Worksheet:",
    worksheet.title
)


# ============================================================
# CELL 10 — READ GOOGLE SHEET
# ============================================================

records = worksheet.get_all_records()

df = pd.DataFrame(
    records
)

print("\n============================================")
print("GOOGLE SHEET")
print("============================================")

print(
    "Total records:",
    len(df)
)

print(
    "Columns:",
    df.columns.tolist()
)


# ============================================================
# CELL 11 — REQUIRED COLUMNS
# ============================================================

REQUIRED_COLUMNS = [
    "image_id",
    "wall_type",
    "indoor",
    "description_en",
    "description_bn",
    "color_idea"
]

missing_columns = [
    col
    for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )

print(
    "✅ Required columns verified"
)


# ============================================================
# CELL 12 — CLEAN TEXT
# ============================================================

df["image_id"] = (
    df["image_id"]
    .astype(str)
    .str.strip()
)


df["description_en"] = (
    df["description_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)


df["description_bn"] = (
    df["description_bn"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Combined text
df["combined_description"] = (
    df["description_en"]
    + " "
    + df["description_bn"]
)


print("✅ Text cleaned")


# ============================================================
# CELL 13 — DUPLICATE IMAGE IDs
# ============================================================

duplicate_mask = (
    df["image_id"]
    .duplicated(
        keep=False
    )
)

duplicate_df = df[
    duplicate_mask
].copy()

duplicate_ids = sorted(
    duplicate_df[
        "image_id"
    ]
    .unique()
    .tolist()
)


print("\n============================================")
print("DUPLICATE IMAGE ID CHECK")
print("============================================")

print(
    "Duplicate IDs:",
    len(duplicate_ids)
)


if duplicate_ids:

    print("\nDuplicates:")

    for item in duplicate_ids:
        print(item)

else:

    print(
        "✅ No duplicate image IDs"
    )


duplicate_df.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "duplicate_image_ids.csv"
    ),
    index=False
)


# ============================================================
# CELL 14 — INDEX DRIVE IMAGES
# ============================================================

image_index = {}

for root, dirs, files in os.walk(
    IMAGE_FOLDER
):

    for filename in files:

        ext = os.path.splitext(
            filename
        )[1].lower()

        if ext not in SUPPORTED_EXTENSIONS:
            continue

        image_id = os.path.splitext(
            filename
        )[0].strip()

        full_path = os.path.join(
            root,
            filename
        )

        image_index.setdefault(
            image_id,
            []
        ).append(
            full_path
        )


print("\n============================================")
print("DRIVE IMAGE INDEX")
print("============================================")

print(
    "Unique image IDs:",
    len(image_index)
)


# ============================================================
# CELL 15 — MATCH SHEET WITH DRIVE
# ============================================================

df["image_path"] = df[
    "image_id"
].map(
    lambda x:
    image_index[x][0]
    if x in image_index
    else np.nan
)


# ============================================================
# CELL 16 — MISSING IMAGES
# ============================================================

missing_df = df[
    df["image_path"].isna()
].copy()


print("\n============================================")
print("MISSING IMAGES")
print("============================================")

print(
    "Missing:",
    len(missing_df)
)


if len(missing_df) == 0:

    print(
        "✅ No missing images"
    )

else:

    for image_id in missing_df[
        "image_id"
    ]:

        print(image_id)


missing_df[
    REQUIRED_COLUMNS
].to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "missing_images.csv"
    ),
    index=False
)


# ============================================================
# CELL 17 — CORRUPTION CHECK
# ============================================================

matched_df = df[
    df["image_path"].notna()
].copy()


corrupted_records = []


print(
    "\nChecking image integrity..."
)


for _, row in matched_df.iterrows():

    path = row["image_path"]

    try:

        with Image.open(
            path
        ) as img:

            img.verify()

        with Image.open(
            path
        ) as img:

            img.load()

    except Exception as e:

        corrupted_records.append({

            "image_id":
                row["image_id"],

            "image_path":
                path,

            "error":
                str(e)
        })


corrupted_df = pd.DataFrame(
    corrupted_records
)


print("\n============================================")
print("CORRUPTED IMAGES")
print("============================================")

print(
    "Corrupted:",
    len(corrupted_df)
)


if len(corrupted_df) == 0:

    print(
        "✅ No corrupted images"
    )

else:

    print(
        corrupted_df
    )


corrupted_df.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "corrupted_images.csv"
    ),
    index=False
)


# ============================================================
# CELL 18 — VALID DATASET
# ============================================================

corrupted_ids = set(
    corrupted_df[
        "image_id"
    ]
    .tolist()
)


valid_df = matched_df[
    ~matched_df[
        "image_id"
    ].isin(
        corrupted_ids
    )
].copy()


valid_df = valid_df.reset_index(
    drop=True
)


print("\n============================================")
print("VALID DATASET")
print("============================================")

print(
    "Valid images:",
    len(valid_df)
)


# ============================================================
# CELL 19 — EXACT / NEAR DUPLICATE IMAGE CHECK
# ============================================================

def file_md5(
    path,
    chunk_size=1024 * 1024
):

    md5 = hashlib.md5()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


hash_to_ids = {}

for _, row in valid_df.iterrows():

    try:

        image_hash = file_md5(
            row["image_path"]
        )

        hash_to_ids.setdefault(
            image_hash,
            []
        ).append(
            row["image_id"]
        )

    except:

        pass


exact_duplicates = {
    h: ids
    for h, ids in hash_to_ids.items()
    if len(ids) > 1
}


print("\n============================================")
print("EXACT DUPLICATE IMAGE CHECK")
print("============================================")

print(
    "Duplicate file groups:",
    len(exact_duplicates)
)


with open(
    os.path.join(
        OUTPUT_FOLDER,
        "exact_duplicate_groups.json"
    ),
    "w"
) as f:

    json.dump(
        exact_duplicates,
        f,
        indent=4
    )


# ============================================================
# CELL 20 — IMAGE DIMENSIONS
# ============================================================

widths = []
heights = []
modes = []


for path in valid_df[
    "image_path"
]:

    with Image.open(
        path
    ) as img:

        widths.append(
            img.width
        )

        heights.append(
            img.height
        )

        modes.append(
            img.mode
        )


valid_df["width"] = widths

valid_df["height"] = heights

valid_df["mode"] = modes


# ============================================================
# CELL 21 — SEMANTIC EVENT / DECORATION GROUPING
#
# Uses BOTH English + Bangla descriptions.
#
# Same / highly similar descriptions are considered
# possible same decoration/event.
# ============================================================

print("\n============================================")
print("DESCRIPTION-BASED GROUPING")
print("============================================")


# Multilingual model
model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)


texts = valid_df[
    "combined_description"
].tolist()


# Generate embeddings
embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)


# Cosine similarity
similarity_matrix = cosine_similarity(
    embeddings
)


n = len(valid_df)


# ============================================================
# Union-Find
# ============================================================

parent = list(
    range(n)
)


def find(
    x
):

    while parent[x] != x:

        parent[x] = parent[
            parent[x]
        ]

        x = parent[x]

    return x


def union(
    a,
    b
):

    root_a = find(a)

    root_b = find(b)

    if root_a != root_b:

        parent[root_b] = root_a


# ============================================================
# Group similar descriptions
# ============================================================

for i in range(n):

    for j in range(
        i + 1,
        n
    ):

        similarity = (
            similarity_matrix[i, j]
        )

        if (
            similarity
            >= DESCRIPTION_SIMILARITY_THRESHOLD
        ):

            union(
                i,
                j
            )


# ============================================================
# Create group IDs
# ============================================================

root_to_group = {}

group_counter = 1

group_ids = []


for i in range(n):

    root = find(i)

    if root not in root_to_group:

        root_to_group[root] = (
            f"group_{group_counter:03d}"
        )

        group_counter += 1

    group_ids.append(
        root_to_group[root]
    )


valid_df["group_id"] = group_ids


print(
    "Description groups:",
    valid_df[
        "group_id"
    ].nunique()
)


# ============================================================
# CELL 22 — SAVE GROUP REVIEW
# ============================================================

group_review = valid_df[
    [
        "image_id",
        "group_id",
        "description_en",
        "description_bn"
    ]
].copy()


group_review.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "group_review.csv"
    ),
    index=False
)


print(
    "✅ group_review.csv saved"
)


# ============================================================
# CELL 23 — GROUP SIZE CHECK
# ============================================================

group_sizes = (
    valid_df[
        "group_id"
    ]
    .value_counts()
)


print("\n============================================")
print("GROUP INFORMATION")
print("============================================")

print(
    "Total groups:",
    len(group_sizes)
)

print(
    "Largest group:",
    group_sizes.max()
)

print(
    "Groups with multiple images:",
    int(
        (group_sizes > 1)
        .sum()
    )
)


# ============================================================
# CELL 24 — GROUP-AWARE TRAIN/VAL/TEST SPLIT
#
# IMPORTANT:
# The same group will never be split across
# train / validation / test.
# ============================================================

groups = valid_df[
    "group_id"
].values


# ------------------------------------------------------------
# First: Train = 80%
# Remaining = 20%
# ------------------------------------------------------------

gss1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)


train_idx, temp_idx = next(
    gss1.split(
        valid_df,
        groups=groups
    )
)


train_df = valid_df.iloc[
    train_idx
].copy()


temp_df = valid_df.iloc[
    temp_idx
].copy()


# ------------------------------------------------------------
# Remaining 20%
# Split into 10% validation + 10% test
# ------------------------------------------------------------

temp_groups = temp_df[
    "group_id"
].values


gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)


val_idx, test_idx = next(
    gss2.split(
        temp_df,
        groups=temp_groups
    )
)


val_df = temp_df.iloc[
    val_idx
].copy()


test_df = temp_df.iloc[
    test_idx
].copy()


# Reset indices
train_df = train_df.reset_index(
    drop=True
)

val_df = val_df.reset_index(
    drop=True
)

test_df = test_df.reset_index(
    drop=True
)


# ============================================================
# CELL 25 — SPLIT STATISTICS
# ============================================================

print("\n============================================")
print("GROUP-AWARE SPLIT")
print("============================================")

print(
    "Total:",
    len(valid_df)
)

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)


print("\nActual ratios:")

print(
    "Train:",
    round(
        len(train_df)
        / len(valid_df),
        4
    )
)

print(
    "Validation:",
    round(
        len(val_df)
        / len(valid_df),
        4
    )
)

print(
    "Test:",
    round(
        len(test_df)
        / len(valid_df),
        4
    )
)


# ============================================================
# CELL 26 — CHECK GROUP LEAKAGE
# ============================================================

train_groups = set(
    train_df[
        "group_id"
    ]
)

val_groups = set(
    val_df[
        "group_id"
    ]
)

test_groups = set(
    test_df[
        "group_id"
    ]
)


train_val = (
    train_groups
    & val_groups
)

train_test = (
    train_groups
    & test_groups
)

val_test = (
    val_groups
    & test_groups
)


print("\n============================================")
print("GROUP LEAKAGE CHECK")
print("============================================")

print(
    "Train ∩ Validation:",
    len(train_val)
)

print(
    "Train ∩ Test:",
    len(train_test)
)

print(
    "Validation ∩ Test:",
    len(val_test)
)


if (
    len(train_val) == 0
    and
    len(train_test) == 0
    and
    len(val_test) == 0
):

    print(
        "✅ NO group leakage!"
    )

else:

    print(
        "⚠️ Group leakage detected!"
    )


# ============================================================
# CELL 27 — ALSO CHECK IMAGE ID OVERLAP
# ============================================================

train_ids = set(
    train_df["image_id"]
)

val_ids = set(
    val_df["image_id"]
)

test_ids = set(
    test_df["image_id"]
)


print("\n============================================")
print("IMAGE ID LEAKAGE CHECK")
print("============================================")

print(
    "Train ∩ Validation:",
    len(
        train_ids
        & val_ids
    )
)

print(
    "Train ∩ Test:",
    len(
        train_ids
        & test_ids
    )
)

print(
    "Validation ∩ Test:",
    len(
        val_ids
        & test_ids
    )
)


# ============================================================
# CELL 28 — SAVE SPLIT CSV FILES
# ============================================================

train_df.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "train.csv"
    ),
    index=False
)


val_df.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "validation.csv"
    ),
    index=False
)


test_df.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "test.csv"
    ),
    index=False
)


print(
    "\n✅ train.csv saved"
)

print(
    "✅ validation.csv saved"
)

print(
    "✅ test.csv saved"
)


# ============================================================
# CELL 29 — IMAGE PREPROCESSING
#
# 224x224
# RGB
# ImageNet normalization
# ============================================================

IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)


# ------------------------------------------------------------
# Training transform
#
# Augmentation ONLY here
# ------------------------------------------------------------

train_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.Lambda(
        lambda img:
        img.convert("RGB")
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=8
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.0,
        hue=0.0
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ------------------------------------------------------------
# Validation/Test transform
#
# NO augmentation
# ------------------------------------------------------------

eval_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.Lambda(
        lambda img:
        img.convert("RGB")
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


print(
    "\n✅ Train transform created"
)

print(
    "✅ Validation/Test transform created"
)


# ============================================================
# CELL 30 — REUSABLE PYTORCH DATASET
# ============================================================

from torch.utils.data import Dataset


class WallDecorationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe.reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image = Image.open(
            row["image_path"]
        ).convert(
            "RGB"
        )


        if self.transform:

            image = self.transform(
                image
            )


        # ----------------------------------------------------
        # We keep all original metadata.
        # No label is changed.
        # ----------------------------------------------------

        return {

            "image":
                image,

            "image_id":
                row["image_id"],

            "wall_type":
                row["wall_type"],

            "indoor":
                row["indoor"],

            "description_en":
                row["description_en"],

            "description_bn":
                row["description_bn"],

            "color_idea":
                row["color_idea"],

            "group_id":
                row["group_id"]
        }


# Create datasets
train_dataset = WallDecorationDataset(
    train_df,
    train_transform
)

val_dataset = WallDecorationDataset(
    val_df,
    eval_transform
)

test_dataset = WallDecorationDataset(
    test_df,
    eval_transform
)


print("\n✅ PyTorch datasets created")

print(
    "Train dataset:",
    len(train_dataset)
)

print(
    "Validation dataset:",
    len(val_dataset)
)

print(
    "Test dataset:",
    len(test_dataset)
)


# ============================================================
# CELL 31 — BEFORE / AFTER PREPROCESSING
# ============================================================

def denormalize(
    tensor
):

    mean = torch.tensor(
        IMAGENET_MEAN
    ).view(
        3,
        1,
        1
    )

    std = torch.tensor(
        IMAGENET_STD
    ).view(
        3,
        1,
        1
    )

    image = (
        tensor.cpu()
        * std
        + mean
    )

    image = torch.clamp(
        image,
        0,
        1
    )

    return image.permute(
        1,
        2,
        0
    ).numpy()


sample_count = min(
    6,
    len(valid_df)
)


sample_df = valid_df.sample(
    sample_count,
    random_state=SEED
)


fig, axes = plt.subplots(
    sample_count,
    2,
    figsize=(
        8,
        4 * sample_count
    )
)


if sample_count == 1:
    axes = np.array([
        axes
    ])


for i, (_, row) in enumerate(
    sample_df.iterrows()
):

    original = Image.open(
        row["image_path"]
    ).convert(
        "RGB"
    )


    processed = eval_transform(
        original
    )


    axes[i, 0].imshow(
        original
    )

    axes[i, 0].set_title(
        f"Original\n{row['image_id']}"
    )

    axes[i, 0].axis(
        "off"
    )


    axes[i, 1].imshow(
        denormalize(
            processed
        )
    )

    axes[i, 1].set_title(
        "224x224 RGB + ImageNet normalization"
    )

    axes[i, 1].axis(
        "off"
    )


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_FOLDER,
        "before_after.png"
    ),
    dpi=150,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# CELL 32 — AUGMENTATION VISUALIZATION
# ============================================================

fig, axes = plt.subplots(
    sample_count,
    4,
    figsize=(
        12,
        3 * sample_count
    )
)


if sample_count == 1:
    axes = np.array([
        axes
    ])


for i, (_, row) in enumerate(
    sample_df.iterrows()
):

    original = Image.open(
        row["image_path"]
    ).convert(
        "RGB"
    )


    for j in range(4):

        augmented = train_transform(
            original
        )


        axes[i, j].imshow(
            denormalize(
                augmented
            )
        )

        axes[i, j].axis(
            "off"
        )

        axes[i, j].set_title(
            f"Training augmentation {j+1}"
        )


plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_FOLDER,
        "augmentation_examples.png"
    ),
    dpi=150,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# CELL 33 — CLASS DISTRIBUTION
# ============================================================

print("\n============================================")
print("WALL TYPE DISTRIBUTION")
print("============================================")

print(
    valid_df[
        "wall_type"
    ].value_counts()
)


# ============================================================
# CELL 34 — SPLIT CLASS DISTRIBUTION
# ============================================================

print("\n============================================")
print("SPLIT DISTRIBUTIONS")
print("============================================")


print("\nTRAIN:")
print(
    train_df[
        "wall_type"
    ].value_counts()
)


print("\nVALIDATION:")
print(
    val_df[
        "wall_type"
    ].value_counts()
)


print("\nTEST:")
print(
    test_df[
        "wall_type"
    ].value_counts()
)


# ============================================================
# CELL 35 — FINAL STATISTICS
# ============================================================

statistics = {

    "google_sheet_records":
        int(len(df)),

    "drive_unique_image_ids":
        int(len(image_index)),

    "missing_images":
        int(len(missing_df)),

    "corrupted_images":
        int(len(corrupted_df)),

    "duplicate_image_ids":
        int(len(duplicate_ids)),

    "exact_duplicate_groups":
        int(len(exact_duplicates)),

    "valid_images":
        int(len(valid_df)),

    "semantic_groups":
        int(
            valid_df[
                "group_id"
            ].nunique()
        ),

    "train_images":
        int(len(train_df)),

    "validation_images":
        int(len(val_df)),

    "test_images":
        int(len(test_df)),

    "train_ratio":
        float(
            len(train_df)
            / len(valid_df)
        ),

    "validation_ratio":
        float(
            len(val_df)
            / len(valid_df)
        ),

    "test_ratio":
        float(
            len(test_df)
            / len(valid_df)
        ),

    "description_similarity_threshold":
        DESCRIPTION_SIMILARITY_THRESHOLD,

    "image_size":
        [224, 224],

    "rgb":
        True,

    "imagenet_mean":
        list(IMAGENET_MEAN),

    "imagenet_std":
        list(IMAGENET_STD),

    "train_augmentation":
        True,

    "validation_test_augmentation":
        False,

    "random_seed":
        SEED
}


print("\n")
print("================================================")
print("FINAL DATASET STATISTICS")
print("================================================")


for key, value in statistics.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# CELL 36 — SAVE STATISTICS
# ============================================================

with open(
    os.path.join(
        OUTPUT_FOLDER,
        "dataset_statistics.json"
    ),
    "w"
) as f:

    json.dump(
        statistics,
        f,
        indent=4
    )


# ============================================================
# CELL 37 — SAVE PREPROCESSING CONFIG
# ============================================================

preprocessing_config = {

    "source": {

        "google_sheet":
            GOOGLE_SHEET_NAME,

        "worksheet":
            SHEET_NAME,

        "image_folder":
            IMAGE_FOLDER
    },


    "image_preprocessing": {

        "resize":
            [224, 224],

        "convert_to_rgb":
            True,

        "normalization":
            "ImageNet",

        "mean":
            list(IMAGENET_MEAN),

        "std":
            list(IMAGENET_STD)
    },


    "training_augmentation": {

        "horizontal_flip":
            0.5,

        "rotation_degrees":
            8,

        "brightness":
            0.10,

        "contrast":
            0.10
    },


    "evaluation":

        {
            "augmentation":
                False
        },


    "grouping": {

        "method":
            "multilingual sentence embeddings",

        "description_fields":
            [
                "description_en",
                "description_bn"
            ],

        "similarity_threshold":
            DESCRIPTION_SIMILARITY_THRESHOLD
    },


    "split": {

        "train":
            TRAIN_RATIO,

        "validation":
            VAL_RATIO,

        "test":
            TEST_RATIO,

        "group_aware":
            True,

        "random_seed":
            SEED
    }
}


with open(
    os.path.join(
        OUTPUT_FOLDER,
        "preprocessing_config.json"
    ),
    "w"
) as f:

    json.dump(
        preprocessing_config,
        f,
        indent=4
    )


# ============================================================
# CELL 38 — SAVE SHORT PREPROCESSING NOTE
# ============================================================

note = f"""
CAMPUS WALL DECORATION
PREPROCESSING PIPELINE
======================

Data source:
- Google Sheet: {GOOGLE_SHEET_NAME}
- Worksheet: {SHEET_NAME}
- Google Drive image folder: {IMAGE_FOLDER}

Quality checks:
- Image ID matching
- Missing image detection
- Corrupted/unreadable image detection
- Duplicate image ID detection
- Exact duplicate file detection

Preprocessing:
- Resize to 224 x 224
- Convert to RGB
- ImageNet normalization

Training augmentation:
- Random horizontal flip
- Small rotation (8 degrees)
- Slight brightness/contrast adjustment

Validation/Test:
- No augmentation

Event/Decoration grouping:
- Combined English + Bangla descriptions
- Multilingual semantic embeddings
- Similarity threshold = {DESCRIPTION_SIMILARITY_THRESHOLD}

Split:
- Approximately 80% train
- Approximately 10% validation
- Approximately 10% test
- Group-aware split to reduce same-event leakage

Original images:
- NOT modified
- NOT overwritten
- NOT deleted

Saved outputs:
- train.csv
- validation.csv
- test.csv
- group_review.csv
- missing_images.csv
- corrupted_images.csv
- duplicate_image_ids.csv
- exact_duplicate_groups.json
- dataset_statistics.json
- preprocessing_config.json
- before_after.png
- augmentation_examples.png
"""


with open(
    os.path.join(
        OUTPUT_FOLDER,
        "preprocessing_note.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(note)


# ============================================================
# CELL 39 — FINAL OUTPUT LIST
# ============================================================

print("\n")
print("================================================")
print("OUTPUT FILES")
print("================================================")


for filename in sorted(
    os.listdir(
        OUTPUT_FOLDER
    )
):

    print(
        filename
    )


# ============================================================
# FINAL
# ============================================================

print("\n")
print("================================================")
print("🎉 PREPROCESSING PIPELINE COMPLETED!")
print("================================================")

print(
    "Original Drive images were NOT changed."
)

print(
    "Google Sheet was used directly."
)

print(
    "Group-aware train/validation/test split created."
)

print(
    "Training-only augmentation configured."
)

print(
    "All reusable metadata and statistics saved."
)

print(
    f"\nOutput folder:\n{OUTPUT_FOLDER}"
)